# Vehicles Processing

This notebook cleans the Milano vehicle fleet raw dataset and writes a mirror cleaned file to data/processed.

Output written to data/processed:
- milan_vehicle_fleet_cleaned.csv

In [1]:
from pathlib import Path

import pandas as pd

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.3f}".format

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = None
for root in candidate_roots:
    if (root / "data" / "raw").exists() and (root / "data" / "processed").exists():
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError("Could not resolve the project root with data/raw and data/processed.")

raw_path = project_root / "data" / "raw" / "milan_vehicle_fleet.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
cleaned_out = processed_dir / "milan_vehicle_fleet_cleaned.csv"

if not raw_path.exists():
    raise FileNotFoundError(f"Missing vehicle fleet raw file: {raw_path}")

print(f"Project root: {project_root}")
print(f"Vehicle fleet raw path: {raw_path}")
print(f"Output cleaned path: {cleaned_out}")

Project root: /Users/faustozamparelli/Documents/Developer/MilanCrash
Vehicle fleet raw path: /Users/faustozamparelli/Documents/Developer/MilanCrash/data/raw/milan_vehicle_fleet.csv
Output cleaned path: /Users/faustozamparelli/Documents/Developer/MilanCrash/data/processed/milan_vehicle_fleet_cleaned.csv


In [2]:
fleet_raw = pd.read_csv(raw_path, sep=";", encoding="utf-8-sig")
fleet_raw.columns = (
    fleet_raw.columns
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"Loaded vehicle fleet rows: {len(fleet_raw)}")
print("Columns:", list(fleet_raw.columns))
display(fleet_raw.head(5))

Loaded vehicle fleet rows: 18
Columns: ['Anno', 'AUTOBUS', 'AUTOCARRI TRASPORTO MERCI', 'AUTOVEICOLI SPECIALI - SPECIFICI', 'AUTOVETTURE', 'MOTOCARRI E QUADRICICLI TRASPORTO MERCI', 'MOTOCICLI', 'MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI', 'RIMORCHI E SEMIRIMORCHI SPECIALI - SPECIFICI', 'RIMORCHI E SEMIRIMORCHI TRASPORTO MERCI', 'TRATTORI STRADALI O MOTRICI', 'ALTRI VEICOLI']


,Anno,AUTOBUS,AUTOCARRI TRASPORTO MERCI,AUTOVEICOLI SPECIALI - SPECIFICI,AUTOVETTURE,MOTOCARRI E QUADRICICLI TRASPORTO MERCI,MOTOCICLI,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,RIMORCHI E SEMIRIMORCHI SPECIALI - SPECIFICI,RIMORCHI E SEMIRIMORCHI TRASPORTO MERCI,TRATTORI STRADALI O MOTRICI,ALTRI VEICOLI
0,2004,2658,65386,10574,739121,1393,115286,370,14837,3796,2149,0
1,2005,2876,66309,10719,739537,1297,123511,451,14944,3873,2146,0
2,2006,2766,66566,10677,736805,1234,129966,529,14754,3972,2121,0
3,2007,2776,66206,10846,726896,1207,135923,617,14681,4138,2085,0
4,2008,3016,66933,11127,723932,1172,140699,694,14735,4412,2205,3


In [3]:
fleet_clean = fleet_raw.copy()

fleet_clean["Anno"] = pd.to_numeric(fleet_clean["Anno"], errors="coerce").astype("Int64")

for col in [c for c in fleet_clean.columns if c != "Anno"]:
    fleet_clean[col] = pd.to_numeric(fleet_clean[col], errors="coerce").round().astype("Int64")

rows_before = len(fleet_raw)
fleet_clean = fleet_clean.drop_duplicates().sort_values(["Anno"], kind="mergesort").reset_index(drop=True)

fleet_clean.to_csv(cleaned_out, index=False)

qc = pd.Series(
    {
        "rows_before": int(rows_before),
        "rows_after": int(fleet_clean.shape[0]),
        "duplicates_removed": int(rows_before - fleet_clean.shape[0]),
        "year_min": int(fleet_clean["Anno"].min()) if fleet_clean["Anno"].notna().any() else pd.NA,
        "year_max": int(fleet_clean["Anno"].max()) if fleet_clean["Anno"].notna().any() else pd.NA,
        "missing_cells": int(fleet_clean.isna().sum().sum()),
    }
)

print(f"Saved: {cleaned_out}")
display(qc.to_frame("value"))
display(fleet_clean.head(10))

Saved: /Users/faustozamparelli/Documents/Developer/MilanCrash/data/processed/milan_vehicle_fleet_cleaned.csv


,value
rows_before,18
rows_after,18
duplicates_removed,0
year_min,2004
year_max,2022
missing_cells,0


,Anno,AUTOBUS,AUTOCARRI TRASPORTO MERCI,AUTOVEICOLI SPECIALI - SPECIFICI,AUTOVETTURE,MOTOCARRI E QUADRICICLI TRASPORTO MERCI,MOTOCICLI,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,RIMORCHI E SEMIRIMORCHI SPECIALI - SPECIFICI,RIMORCHI E SEMIRIMORCHI TRASPORTO MERCI,TRATTORI STRADALI O MOTRICI,ALTRI VEICOLI
0,2004,2658,65386,10574,739121,1393,115286,370,14837,3796,2149,0
1,2005,2876,66309,10719,739537,1297,123511,451,14944,3873,2146,0
2,2006,2766,66566,10677,736805,1234,129966,529,14754,3972,2121,0
3,2007,2776,66206,10846,726896,1207,135923,617,14681,4138,2085,0
4,2008,3016,66933,11127,723932,1172,140699,694,14735,4412,2205,3
5,2010,3137,65939,10964,716454,1088,149016,765,2293,3781,2290,3
6,2011,3074,65866,10866,724450,1090,152858,783,2250,3910,2278,1
7,2012,2866,65104,10814,716094,1075,155142,1036,2333,3913,2249,0
8,2013,2766,63539,10515,701301,1077,156736,1128,2294,3995,2264,0
9,2014,2679,62391,10488,686379,1094,157808,1227,2371,4039,2340,0


## Fleet Time-Series Readiness Check

This section adds year continuity and large year-over-year change diagnostics for downstream trend analysis.

In [4]:
fleet_ts = fleet_clean.sort_values("Anno").copy()
years = fleet_ts["Anno"].dropna().astype(int).tolist()
missing_years = sorted(set(range(min(years), max(years) + 1)) - set(years)) if years else []

num_cols = [c for c in fleet_ts.columns if c != "Anno"]
yoy = fleet_ts[["Anno", *num_cols]].copy()
for col in num_cols:
    yoy[f"yoy_pct_{col}"] = 100 * yoy[col].pct_change()

# Surface notable jumps for analyst review.
jump_rows = []
for col in num_cols:
    c = f"yoy_pct_{col}"
    sub = yoy.loc[yoy[c].abs() >= 15, ["Anno", c]].copy()
    for _, row in sub.iterrows():
        jump_rows.append({"Anno": int(row["Anno"]), "metric": col, "yoy_pct": float(row[c])})

jumps_df = pd.DataFrame(jump_rows).sort_values(["Anno", "metric"]).reset_index(drop=True) if jump_rows else pd.DataFrame(columns=["Anno", "metric", "yoy_pct"])

qc_ts = pd.Series(
    {
        "year_min": min(years) if years else pd.NA,
        "year_max": max(years) if years else pd.NA,
        "n_years": len(years),
        "missing_years_in_span": len(missing_years),
        "large_yoy_jumps_abs_ge_15pct": int(len(jumps_df)),
    }
)

print("Fleet time-series profile:")
display(qc_ts.to_frame("value"))
print("Missing years in span:", missing_years)
print("Large YoY jumps (absolute >= 15%):")
display(jumps_df.head(40))

Fleet time-series profile:


,value
year_min,2004
year_max,2022
n_years,18
missing_years_in_span,1
large_yoy_jumps_abs_ge_15pct,10


Missing years in span: [2009]
Large YoY jumps (absolute >= 15%):


,Anno,metric,yoy_pct
0,2005,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,21.892
1,2006,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,17.295
2,2007,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,16.635
3,2008,ALTRI VEICOLI,inf
4,2010,RIMORCHI E SEMIRIMORCHI SPECIALI - SPECIFICI,-84.438
5,2011,ALTRI VEICOLI,-66.667
6,2012,ALTRI VEICOLI,-100.000
7,2012,MOTOVEICOLI E QUADRICICLI SPECIALI - SPECIFICI,32.312
8,2018,AUTOBUS,15.057
9,2021,AUTOBUS,43.339
